# Agilent BioTek MultiFlo FX dispenser quickstart

The MultiFlo FX is a bulk dispenser. It fills wells from two kinds of pump — syringes, which meter
a volume precisely, and peristaltic pumps, which push fluid through a tubing cassette — and can
wash a plate through a strip washer manifold rather than the plate wash manifold its washer
siblings carry. Which of those it actually has is a matter of what somebody fitted and which
firmware image is installed, and it reports both.

This quickstart connects to the dispenser, reads what it is and what it has fitted, works out which
firmware variant that leaves it running, tells it which plate is on its carrier, primes and
dispenses from both syringes and both pumps, explains cassettes and where in the well a step works,
runs a protocol file, and disconnects.

| Property | Value |
|---|---|
| Communication | A serial port, or USB through an FTDI interface |
| Serial parameters | 38400 baud, 8 data bits, 2 stop bits, no parity, no flow control |
| Operations | Syringe priming and dispensing, peristaltic priming, purging and dispensing, peristaltic and strip washing, shake and soak |
| Plate formats | 6 to 1536 wells, resolved from the PyLabRobot plate resource |
| Syringes | A and B, each with two selectable bottles; both at once on the manifolds that allow it |
| Peristaltic pumps | Primary and secondary, one tubing cassette each |
| Dispenser reach | Depth 1-1500 motor steps; across the well ±125 for the syringes, ±400 for the pumps; along the well ±40 |
| Protocol files | `.LHC` protocol files are read, checked and run |
```{warning}
Follow the manufacturer's installation, fluid-handling and safety instructions. Priming, purging
and dispensing all move fluid: the bottles must be full and the waste bottle empty enough before
anything in this notebook runs.
```

```{device-card} biotek-multiflo-fx
```

## How it communicates

PyLabRobot frames each command as an 11-byte header and a payload, writes it to the instrument,
reads back an acknowledgement and the reply, and turns a non-zero status into a typed exception.
Which of the two transports carries those bytes is decided by the port string alone, and nothing
above that point knows which one it got.

Operations that move fluid do not answer when they are done. The driver sends them, then polls the
instrument's run state until it stops reporting a step in progress, which is why every method that
touches the instrument is awaited and can take as long as the physical operation does.

Install PyLabRobot with its serial dependencies, or with the FTDI ones if the dispenser is on USB.

Reading `.LHC` protocol files additionally needs `pycryptodome`, which is not a PyLabRobot
dependency: the file format is encrypted, and the cipher is not in the standard library. Leave it
out if you only build protocols in Python — `read()` raises a `RuntimeError` telling you to install
it if you later try to read a file without it.

In [ ]:
%pip install "pylabrobot[serial]" pycryptodome

# On USB, install the FTDI dependencies instead: "pylabrobot[ftdi]".

## Physical setup and finding the port

Install, plumb and power the dispenser according to the manufacturer's instructions. Fit the tubing
cassettes into the pumps you intend to use, connect each syringe to the bottle it draws from,
connect the waste bottle, and connect the instrument to the computer.

Then find the port string:

- **Serial.** Pass the operating system's own name for the port: `COM3` on Windows,
  `/dev/ttyUSB0` or `/dev/ttyS4` on Linux and macOS. Anything that is not a USB serial number is
  taken to be a serial port and passed through unexamined, so no particular naming pattern is
  required.

- **USB.** List the attached FTDI devices and use the reported serial number:

  ```bash
  python -m pylibftdi.examples.list_devices
  ```

  The port is then `ftdi:<serial>`, for example `ftdi:183193P`. The form the instrument's own
  protocol files record, `USB MultiFlo FX sn:183193P`, is accepted as well.

Keep the carrier and the area around it clear from here on. Nothing in this notebook moves the
carrier before the priming section, but a fault can home the motors at any time.

## Turn on logging

The driver reports what it is doing through the standard library's `logging`, and says nothing
otherwise. Without this cell every step the instrument runs passes silently.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

## See which ports have something behind them

`pyserial` lists every serial port the operating system offers, most of which are kernel
placeholders with no hardware behind them — on Linux the 32 `/dev/ttyS*` entries, which report
their description and hardware id as `n/a`. Skipping those leaves the ports worth trying, and a
USB-serial adapter names the instrument it is wired to, so the dispenser is usually recognisable at
a glance.

This lists serial ports only. An instrument driven through the FTDI transport is found with
`python -m pylibftdi.examples.list_devices` instead — though a device the kernel has bound to its
own FTDI serial driver shows up here too, and can be used either way.

In [ ]:
from serial.tools.list_ports import comports

candidates = [port for port in comports() if port.description != "n/a" or port.vid is not None]

for port in candidates:
    serial_number = f"  sn:{port.serial_number}" if port.serial_number else ""
    print(f"{port.device:16} {port.description}{serial_number}")

if not candidates:
    print("no port has a device behind it; is the dispenser powered and connected?")

## Build the dispenser

Constructing the object opens nothing and touches no hardware. It records which port to use, which
model this is, and what to call the instrument in logs and error messages.

Replace the port with the one found above.

In [ ]:
from pylabrobot.agilent.biotek.lhc import MultiFloFX

# The port is a serial one unless it carries a device serial number: on USB, pass
# "ftdi:YOUR_SERIAL" instead.
device = MultiFloFX(port="/dev/ttyUSB2", name="MultiFlo FX")
device

## Connect

`setup()` opens the link, asks whether anything is listening, and reads the options the instrument
has fitted. That read is not optional: every step is encoded against it and every check measures
against it, so a failure here stops the notebook rather than being carried past.

It raises a `BiotekError` if the port will not open, if nothing answers on it, or if the fitted
options cannot be read.

In [ ]:
await device.setup()

## Ask what answered

**The model is declared, not discovered.** The instrument does not report which model it is, so it
is the class you constructed that decides how every step is encoded and which options are read.
Note that `device.settings.family` is not a check on this — it echoes what was declared, not what is
attached.

What the instrument does report is its serial number, which identifies the individual instrument,
and its firmware version record, whose part number says which instrument the installed firmware
image is built for. A MultiFlo FX reports a part number beginning `126`.

In [ ]:
print("serial number:  ", await device.get_serial_number())

version = await device.get_firmware_version()
print("part number:    ", version.part_number)
print("firmware:       ", version.software_version)
print("data version:   ", version.data_version)

## Read the configuration

What `setup()` read is kept as a read-only record of what somebody fitted to this instrument. It is
read-only because the instrument is: of its whole command vocabulary, almost nothing about the
configuration can be written, so a record that could be edited would only mislead. The one
exception is which cassette sits in which pump, and that is reconciled when a batch opens rather
than held here.

This is the record every check below measures against, and on this model it decides more than on
any other: which pumps exist, whether there are one or two syringes, whether a strip washer is
fitted, and — through the last two flags — which firmware variant the instrument is running.

In [ ]:
settings = device.settings

print("family:              ", settings.family.name)
print("syringe box:         ", settings.syringe_box.name)
print("syringe box size:    ", settings.syringe_box_size.name)
print("syringe manifold:    ", settings.syringe_manifold.name)
print("primary peri pump:   ", settings.peri_pump)
print("secondary peri pump: ", settings.peri_pump_2)
print("strip washer:        ", settings.strip_washer_manifold.name)
print("wash manifold:       ", settings.washer_manifold.name)
print("single well enabled: ", settings.single_well_enabled)
print("peri wash enabled:   ", settings.peri_wash_enabled)
print("wider dispense offsets:", settings.advanced_dispense_offsets)

### Two syringes and two pumps

The two syringes are one box: `syringe_box_size` is `DOUBLE` for a box driving A and B, and
`SINGLE` for one driving A alone. Which bottle each syringe draws from is chosen per step, so a
double box gives four combinations and a step that runs both syringes names the pairing.

Running **both at once** takes more than the box, though: it is the fitted *manifold* that decides,
and only four of them can — the 8-tube, the 16/7-tube and both 32-tube manifolds. On the plain
16-tube manifold each step drives one syringe, and a step asking for both is refused saying so. A
prime is never both regardless: it is checked against a 16-tube manifold whatever is fitted, so
that a prime is never rejected for the manifold it will actually run on.

The two peristaltic pumps are separate options, `peri_pump` and `peri_pump_2`, and each holds one
tubing cassette. A step names the pump it wants and the cassette it needs; asking for a pump that
is not fitted is refused with a reason rather than quietly running on the other one.

## Ask what it can run, and why that is not just the hardware

A model can be built to run a fixed set of operations, a particular instrument runs the subset its
fitted hardware supports — and on this model the **firmware image narrows it again**. Its firmware
ships in three variants, and the instrument reports which one it is running through the two flags
read above:

| Variant | Reported by | Offers |
|---|---|---|
| Basic | neither flag set | peristaltic steps, syringe steps, strip washer steps, shake and soak |
| Random access | `single_well_enabled` | peristaltic steps, syringe steps, dispensing into individually chosen wells, shake and soak |
| PeriWash | `peri_wash_enabled` | peristaltic steps, the peristaltic wash pair, shake and soak |

Read that table for what it costs: **the PeriWash variant has no syringe steps at all**, and
neither of the other two has the peristaltic wash pair. A step type the installed firmware does not
know is refused before anything moves, with a code of its own for exactly that. If both flags are
set the instrument is taken to be running the random-access variant.

`get_available_steps()` has already applied all of it, which makes it the answer to "why was my
step refused" before you have written the step.

In [ ]:
for step_type in device.get_available_steps():
    print(step_type.name)

## Tell it which plate is on the carrier

Nothing runs until a plate has been set. Every step carries the height it works at, measured from
the nominal heights of the format on the carrier, so without one there is nothing to measure from
and the driver raises `RejectedError` rather than guessing.

The format is resolved from the PyLabRobot plate resource itself — its columns, its rows, and how
deep its wells are. Labware that does not land on exactly one of the formats this model works is an
error naming the candidates, never a nearest fit. This model works the widest range in the family,
from 6-well plates to 1536-well ones.

In [ ]:
from pylabrobot.resources import Greiner_384_wellplate_28ul_Fb

plate = Greiner_384_wellplate_28ul_Fb(name="plate")
device.set_plate(plate)

print(device.plate)

Some formats are never resolved from a resource, because they share their column and row count with
an ordinary plate and differ in something the resource does not carry — a well shape, a flange,
a tube, or that it is calibration labware. Name one of those outright, and use the same argument
when a plate is to be worked as something other than what it resolves to.

In [ ]:
from pylabrobot.agilent.biotek.lhc.enums.plates.plate_type import PlateType

device.set_plate(plate, plate_type=PlateType.PLATE_384_WELL_PCR)
print(device.plate)

device.set_plate(plate)  # back to what it resolves to on its own

## Check before running

`can_run()` measures steps against the instrument as it is now — what is fitted, which firmware
variant is installed, which plate is on the carrier, and what the plate will accept — and touches
nothing. It is what `run_protocol()` does first, so calling it yourself is how you see a refusal
without moving anything.

The report is truthy when everything can run, and prints as the list of what cannot.

In [ ]:
from pylabrobot.agilent.biotek.lhc.protocols.steps.steps.peri_prime import PeriPrime

report = await device.can_run([PeriPrime(volume=300, peri_pump="Primary")])
print(report)
print("can run:", bool(report))

A volume the syringe will not meter is refused with the range it would accept. The floor moves with
the plate on the carrier and the flow rate, so it is worth reading off the message rather than
memorising: into a 384-well plate at flow rate 2 it is 10 µL, and at the slowest rate 40 µL.

In [ ]:
from pylabrobot.agilent.biotek.lhc.protocols.steps.steps.syringe_dispense import SyringeDispense

print(await device.can_run([SyringeDispense(volume=5, syringe="B")]))

## Prime the syringes

Priming draws fluid through a syringe until its lines are full. It is the safest operation to try
first — nothing is dispensed into the plate — but it does move fluid, so check the bottles before
running this cell.

A prime drives **one** syringe: unlike a dispense, it cannot run both at once, and a prime naming
`"Both"` is refused. Priming each in turn inside one batch is how both get done without homing the
motors twice.

In [ ]:
async with device.batch():
    await device.syringe_dispenser.prime(volume=5_000, syringe="A", syringe_bottle="A1", cycles=2)
    await device.syringe_dispenser.prime(volume=5_000, syringe="B", syringe_bottle="B1", cycles=2)

## Dispense from each syringe

**This dispenses into the plate**, so put a plate on the carrier that you are willing to fill.

Each syringe draws from one of its two bottles, named per step. Volumes are per well in µL.

In [ ]:
async with device.batch(home_on_close=True):
    await device.syringe_dispenser.dispense(volume=20, syringe="A", syringe_bottle="A1")
    await device.syringe_dispenser.dispense(volume=20, syringe="B", syringe_bottle="B1")

Both syringes can also dispense at once, on the manifolds that allow it. The step then names the
pairing rather than a single bottle: `"A1B1"` draws syringe A from its first bottle and syringe B
from its first, `"A2B1"` from A's second and B's first.

`"Both"` does not put both fluids in every well. Nothing in the step divides the plate — one
volume, one flow rate and one column selection cover the pair — but each syringe feeds its own
tubes in the manifold, so the wells divide between them. Observed on a MultiFlo FX with the
16-tube 7° manifold, watched with the lines dry: the columns fall in pairs, syringe B taking 3-4,
7-8 and 11-12 and syringe A the rest. That mapping belongs to the fitted manifold and the plate
format rather than to the protocol — the vendor's own library carries no per-syringe well
selection at all — so confirm it on your instrument before a step depends on it.

LHC offers the same choice as the `Syringe:` radio group — `A`, `B`, `Both` — in its Syringe
Dispense Step dialog, alongside the bottle list that reads `A1 + B1` for the pairing, and says no
more about it there than the name does here.

The cell below asks the fitted manifold first, so it is safe to run on any instrument.

In [ ]:
from pylabrobot.agilent.biotek.lhc.enums.instrument.syringe_manifold import SyringeManifold

BOTH_AT_ONCE = {
    SyringeManifold.TUBE_8,
    SyringeManifold.TUBE_16_7,
    SyringeManifold.TUBE_32_SMALL_BORE,
    SyringeManifold.TUBE_32_LARGE_BORE,
}

if device.settings.syringe_manifold in BOTH_AT_ONCE:
    await device.syringe_dispenser.dispense(volume=20, syringe="Both", syringe_bottle="A1B1")
else:
    print(
        f"a {device.settings.syringe_manifold.name} manifold drives one syringe per step; "
        "run the two dispenses above instead"
    )

One thing to know before writing a protocol: **a syringe is committed to one bottle for the whole
run**. Two steps drawing syringe A from different bottles are refused — on the second of the pair,
which is where the conflict becomes visible — because nothing switches the bottle mid-run.

Where a syringe is plumbed to one bottle and nothing else, the choice does not arise and
`syringe_bottle` can be left out: it defaults to `"A1"`. It defaults to that whichever syringe the
step drives, so a step on B names `"B1"` for itself. Switching bottles within a run at all takes
the syringe buffer-switching valve module, which only the EL406 is asked about, so on this model
the commitment above always holds.

In [ ]:
print(
    await device.can_run(
        [
            SyringeDispense(volume=20, syringe="A", syringe_bottle="A1"),
            SyringeDispense(volume=20, syringe="A", syringe_bottle="A2"),
        ]
    )
)

## Prime and purge the peristaltic pumps

A peristaltic pump pushes fluid through a tubing cassette. Priming fills the tubing; purging empties
it, running fluid to waste rather than into the plate — which is what you want at the end of a run
and before a cassette comes out.

Both take either a volume per tube or a duration: give `duration` and the step runs for that many
seconds instead of metering a volume.

In [ ]:
async with device.batch():
    await device.peristaltic_dispenser.prime(volume=300, peri_pump="Primary")
    await device.peristaltic_dispenser.prime(duration=10, peri_pump="Secondary")

Purging is worth doing on its own at the end of a run, and before a cassette comes out: what stays
in the tubing otherwise dries in it.

In [ ]:
await device.peristaltic_dispenser.purge(volume=300, peri_pump="Secondary")

## Dispense from the peristaltic pumps

**This one dispenses into the plate too.** The volume is per tube in µL, and the flow rate is one of
three named speeds rather than a number.

A step may also name the cassette it needs. `"Any"` accepts whatever is fitted; naming `"1uL"`,
`"5uL"` or `"10uL"` is a requirement, and it is checked before anything moves.

In [ ]:
async with device.batch(home_on_close=True):
    await device.peristaltic_dispenser.dispense(volume=20, peri_pump="Primary")
    await device.peristaltic_dispenser.dispense(
        volume=20, flow_rate="Low", cassette_type="5uL", peri_pump="Secondary"
    )

### One cassette per pump

Each pump holds one cassette, and **opening a batch is what makes the hardware match what the
checked protocol asked for** — this is the only model whose peristaltic dispense head can be set
from the host, so the reconciliation happens there rather than by hand.

The rule that follows is that two steps may not want different cassettes in the same pump. Such a
protocol is refused on the second of the pair; the same two steps, one per pump, are fine.

In [ ]:
print(
    "same pump, two cassettes:",
    await device.can_run(
        [
            PeriPrime(volume=300, cassette_type="1uL", peri_pump="Primary"),
            PeriPrime(volume=300, cassette_type="5uL", peri_pump="Primary"),
        ]
    ),
)
print(
    "one per pump:",
    await device.can_run(
        [
            PeriPrime(volume=300, cassette_type="1uL", peri_pump="Primary"),
            PeriPrime(volume=300, cassette_type="5uL", peri_pump="Secondary"),
        ]
    ),
)

## Shake and soak

Shaking and soaking are one step, and either half can be left out by giving it no time: `duration`
shakes, `soak_duration` leaves the plate still afterwards. Both are in seconds.

This is the one operation every variant of the firmware offers, whatever is fitted.

In [ ]:
await device.shake(duration=30, intensity="Medium", axis="X", soak_duration=30)

## Where in the well a step works

Every operation that reaches into the plate takes a `positioning`, and defaults it to the nominal
position for that head over the format on the carrier. Three numbers:

- `z_steps` — how deep the dispense tubes go. **This is a height, not an offset**: it defaults to
  the plate record's own nominal height for the head, and giving a larger number reaches further
  down into the well. On this model it must be 1-1500, whichever head is working.
- `x_steps` — across the well. The range is per head: ±125 from the syringes, ±400 from the
  peristaltic pumps, the strip washer and the peristaltic wash manifolds.
- `y_steps` — along the well, ±40 from the syringes and the pumps, ±99 from the strip washer, the
  peristaltic wash manifolds and a random-access dispense.

The `advanced_dispense_offsets` flag read above widens the syringes to the ±400 and ±99 the rest
already have. It is an instrument setting, not something a step asks for, which is why a syringe
offset that one instrument accepts is refused on another.

The nominal heights come from the plate record, so they change with the plate and differ per head.
On this model the dispensers work at the dispensing height and the peristaltic wash manifold sits
markedly higher.

In [ ]:
print("nominal dispensing height:", device.plate.dispenser_height)
print("nominal aspirating height:", device.plate.manifold_aspirate_height)

To dispense a little higher than nominal — down the side of the well rather than into the middle of
it — build a `Positioning` from the nominal height rather than from a number you have written down,
so the same code stays right when the plate changes.

In [ ]:
from pylabrobot.agilent.biotek.lhc.protocols.steps.step_parts.positioning import Positioning

await device.syringe_dispenser.dispense(
    volume=20,
    syringe="A",
    positioning=Positioning(
        z_steps=device.plate.dispenser_height - 20,  # 20 steps higher in the well
        x_steps=15,  # toward one side
        y_steps=0,
    ),
)

### Why motor steps and not millimetres

PyLabRobot's convention is millimetres, and this is the one place the package deviates from it. The
field names say so outright — `z_steps`, not `z` — because the conversion is not a single number:
it differs per axis, per model, and per head, and only part of it is established.

What is known: the across-the-plate axis is **0.04572 mm per motor step**, confirmed twice over
against a published maximum offset. The depth axis has at least two scales, chosen by a property
that follows the head, and which head takes which is not settled. The along-the-plate axis is not
established at all.

Converting on the strength of that would put a head at the wrong depth on some model, so the
package does not convert. If you need millimetres on your instrument, measure them: drive a known
offset on each axis and see where the tubes go. A step is also what a protocol file stores, which
is what lets a protocol be read, checked and written with no instrument to ask.

## Working part of a plate

Every step that reaches into the plate takes a `columns` selection, and most of them take a `rows`
one as well. Both default to everything, which is why nothing above has named them.

A selection is a `WellMask`: one entry per position, `1` to work it and `0` to skip it.
`from_columns` builds one from column numbers, counted from one, and a list of entries is the way
to write one out by hand. A column selection always has 48 entries whatever the plate holds — the
instrument reads as many as the format on the carrier has, so on a 384-well plate everything past
the 24th is ignored, and naming a column the plate does not have is not an error.

Selecting no column at all is allowed and dispenses nothing, which is the vendor's own behaviour
rather than an oversight. An entry that is neither 0 nor 1 is refused as corrupt data.

In [ ]:
from pylabrobot.agilent.biotek.lhc.protocols.steps.step_parts.masks import WellMask

first_two = WellMask.from_columns([1, 2])
every_other = WellMask.from_columns(range(1, 49, 2))  # columns 1, 3, 5, ...

print("first two selects: ", first_two.selected_count)
print("every other selects:", every_other.selected_count)
print(await device.can_run([SyringeDispense(volume=20, syringe="A", columns=every_other)]))

**This dispenses into the plate**, into the first two columns and nowhere else.

In [ ]:
async with device.batch(home_on_close=True):
    await device.syringe_dispenser.dispense(volume=20, syringe="A", columns=first_two)

Rows go in sections of eight rather than one at a time, and a plate has `rows // 8` of them — one
on a 96-well plate, two on a 384-well, four on a 1536-well. Selecting a section the plate does not
have counts as selecting nothing.

Whether sections can be addressed at all belongs to the head doing the work rather than to the
plate. A head that covers every row in one pass leaves nothing to choose between, and a selection
handed to it does nothing: a 16-tube syringe manifold reaches all sixteen rows of a 384-well plate
at once, so a row selection on a syringe dispense there is inert. It bites where the head covers
less than the plate — the peristaltic pumps, whose cassettes carry eight tubes, and the 8-tube
syringe manifold on a 384-well plate.

Nothing refuses an inert selection, either. Of the steps that carry rows only the peristaltic
dispense, the strip dispense and the peristaltic wash pair check them, and there an empty selection
is refused — where an empty column selection is allowed. Naming `rows` at all is what makes a step
store and send them, and the payload is padded to a fixed length either way, so the selection costs
nothing when it is there.

In [ ]:
print("row sections on this plate:", device.plate.rows // 8)
print("syringe manifold:          ", device.settings.syringe_manifold.name)

# A cassette carries eight tubes, so for the pumps a section is a real choice on this plate.
await device.peristaltic_dispenser.dispense(
    volume=20, peri_pump="Primary", columns=first_two, rows=WellMask([1, 0, 0, 0])
)

On a wash, put the selection on the wash itself rather than on the dispense handed
to it: a sub-step's own masks are dropped, in the protocol file as on the wire.

In [ ]:
from pylabrobot.agilent.biotek.lhc.enums.instrument.strip_washer_manifold import (
    StripWasherManifold,
)
from pylabrobot.agilent.biotek.lhc.protocols.steps.steps.strip_dispense import StripDispense

if device.settings.strip_washer_manifold is not StripWasherManifold.NOT_INSTALLED:
    async with device.batch(home_on_close=True):
        await device.washer.strip_wash(
            cycles=3,
            dispense=StripDispense(volume=25, flow_rate=3),
            columns=first_two,
        )
else:
    print("no strip washer fitted; nothing to do")

## Dispensing into individually chosen wells

On the random-access variant — the one reporting `single_well_enabled` — a peristaltic dispense can
address wells one at a time instead of the whole plate. Giving a per-well volume map, a dispense
head, or both makes the step that kind of dispense, which is a different payload and a different
command.

Two rules come with it, and both are checked rather than assumed: the step runs on the **secondary**
pump when two are fitted, and the head has to suit the plate — `"1 tube to 1 well"` on a 384-well
plate, where `"8 tubes to 8 wells"` is refused.

One oddity in how this is reported, worth knowing before concluding the option is missing: the
instrument is only asked whether the random-access dispenser is fitted once its **strip washer
hardware** answers. On an instrument that reports no strip washer hardware at all,
`single_well_enabled` reads false whatever is actually fitted, and every random-access dispense is
refused. That is how the vendor's own library asks, so it is what this driver does.

The cell below runs only where the firmware offers it, so it is safe to run on any instrument.

In [ ]:
from pylabrobot.agilent.biotek.lhc.enums.steps.step_type import StepType

if device.settings.single_well_enabled:
    await device.peristaltic_dispenser.dispense(
        volume=20, peri_pump="Secondary", cassette_head="1 tube to 1 well"
    )
else:
    print("this instrument does not run the random-access firmware; nothing to do")

## Gentle medium exchange

On the PeriWash variant — the one reporting `peri_wash_enabled` — the peristaltic pumps drive a pair
of wash manifolds that exchange medium without disturbing what is growing in the well: one
aspirates the spent medium off, the other adds fresh. Both work at the plate's aspirating height,
which is the instrument's own pairing rather than this package's.

Two things to know. A wash cassette cannot share a pump with an ordinary peristaltic step, and a
protocol that asks for both on one pump is refused saying so. And this variant has **no syringe
steps** — if the cells above dispensed from a syringe, this instrument is not running it.

In [ ]:
if device.settings.peri_wash_enabled:
    async with device.batch(home_on_close=True):
        await device.peristaltic_dispenser.wash_aspirate(volume=25, peri_pump="Primary")
        await device.peristaltic_dispenser.wash_dispense(volume=25, peri_pump="Primary")
else:
    print("this instrument does not run the PeriWash firmware; nothing to do")

## The strip washer

Where the washers in this family carry a plate wash manifold, this model washes through a strip
washer manifold — so of the methods on `device.washer` only the strip ones run here, and
`get_available_steps()` above says whether even those do.

The manifold is fitted for a plate format, and it has to match the plate on the carrier: the 96-well
manifold covers 96- and 384-well plates, where a 24-well manifold over either is refused. A wash is
one step rather than a loop — the instrument runs the cycles itself — and its refill needs a volume
of its own, since the step type defaults to none.

In [ ]:
from pylabrobot.agilent.biotek.lhc.enums.instrument.strip_washer_manifold import (
    StripWasherManifold,
)
from pylabrobot.agilent.biotek.lhc.protocols.steps.steps.strip_dispense import StripDispense

if device.settings.strip_washer_manifold is not StripWasherManifold.NOT_INSTALLED:
    async with device.batch(home_on_close=True):
        await device.washer.strip_prime(volume=320, flow_rate=3)
        await device.washer.strip_wash(
            cycles=1, dispense=StripDispense(volume=25, flow_rate=3)
        )
else:
    print("no strip washer fitted; nothing to do")

## Run several operations in one batch

Opening a batch homes the motors, reconciles the cassettes and the dispense head with what the
checked protocol asked for, takes the instrument so that nothing else can interleave a run on it,
and holds it until the block ends. Every operation opens one; doing it once around several
operations — as the cells above have been doing — is what stops all of that happening between each
of them.

`home_on_close=True` drives the transport home before the batch closes. The instrument does not do
this by itself; ask for it when the next thing to touch the plate is a person.

Nesting is allowed and does nothing: an operation called inside an open batch joins it rather than
opening a second one.

In [ ]:
async with device.batch(home_on_close=True):
    await device.peristaltic_dispenser.prime(volume=300, peri_pump="Primary")
    await device.peristaltic_dispenser.dispense(volume=20, peri_pump="Primary")
    await device.shake(duration=30, soak_duration=0)

## Watch a step, and stop it

`get_status()` reports what the instrument is doing, which timed phase a running step is in, and how
many seconds are left in it. It can be called at any time, including while a step is running.

An operation does not return until its step has finished, so pausing or aborting means asking from
somewhere else while it runs. In a notebook that is a task.

In [ ]:
import asyncio

running = asyncio.create_task(
    device.peristaltic_dispenser.prime(duration=30, peri_pump="Primary")
)
await asyncio.sleep(2)

status = await device.get_status()
print("state:    ", status.state.name)
print("activity: ", status.activity.name)
print("remaining:", status.remaining, "s")

Pause holds the step where it is; resume carries on from there.

In [ ]:
from pylabrobot.agilent.biotek.lhc.enums.status.run_state import RunState

status = await device.get_status()
if status.state is not RunState.BUSY:
    print(
        f"the device is {status.state.name}, not running a step -- start the cell above again "
        "and run this one while its step is still going"
    )
else:
    await device.pause()
    await asyncio.sleep(2)
    print((await device.get_status()).state.name)

    await device.resume()
    await running

Abort stops the running step instead. The operation that was waiting for it raises `AbortedError`,
which is a `BiotekError`, so a protocol run ends where it was stopped rather than carrying on to the
next step.

In [ ]:
from pylabrobot.agilent.biotek.lhc.error_handling import AbortedError

running = asyncio.create_task(
    device.peristaltic_dispenser.prime(duration=30, peri_pump="Primary")
)
await asyncio.sleep(5)
await device.abort()

try:
    await running
except AbortedError as error:
    print("stopped:", error)

## Run a protocol file

A `.LHC` protocol file is read into a `Protocol`: what it will run, and everything the file records
alongside it. Reading needs `pycryptodome`, installed at the top of this notebook.

Reading a file never fails on a step it cannot understand — the file's own records are kept as they
are, so a protocol from another model still reads, prints and writes. `build_steps()` is what turns
those records into steps, and it names the one that will not read.

In [ ]:
from pylabrobot.agilent.biotek.lhc import read

protocol = read("path/to/your/protocol.LHC")

print("name:      ", protocol.protocol_name)
print("written by:", protocol.lhc_version)
print("written for:", protocol.instrument_name)
print("plate:     ", protocol.plate_type or protocol.plate_type_number)
print(
    "entries:   ",
    len(protocol.entries),
    "of which",
    len(protocol.device_entries),
    "operate the instrument",
)

for index, step in enumerate(protocol.build_steps()):
    print(f"  step {index}: {type(step).__name__}")

### What the file says about its instrument

A protocol file records the options the instrument had fitted when it was written. Nothing runs
against that record — steps are encoded against the instrument in front of you — and nothing writes
it to the instrument. It is good for exactly one question, worth asking about a file that came from
another machine: was this written for a differently equipped dispenser? On this model that question
has teeth, because a file written for a two-pump instrument will not run on a one-pump one.

`compare_settings()` is truthy when the two agree, and prints as the options that differ. It raises
`ValueError` for a file that carries no such record, which is how the oldest releases wrote one.

In [ ]:
comparison = device.compare_settings(protocol)
print(comparison)
print("same configuration:", bool(comparison))

### Running it

`run_protocol()` checks the protocol, opens one batch around the whole run, sends each step and
polls it to completion. The check is the same `can_run()` from above and happens automatically, so a
protocol that cannot run raises before anything moves.

**This runs whatever the protocol does**, which for most dispenser protocols means filling every
well of the plate on the carrier. Read the steps printed above first.

The entries that sequence a run rather than operate the instrument — delays, loops, remarks — are
not run; the device steps go in file order. They are still there on `protocol.entries` to inspect.

In [ ]:
await device.run_protocol(protocol, home_on_close=True)

Steps built in Python run the same way. `run_protocol()` takes a list of steps as readily as a
protocol, and `run_step()` runs a single one.

In [ ]:
await device.run_protocol(
    [
        PeriPrime(volume=300, peri_pump="Primary"),
        PeriPrime(volume=300, peri_pump="Secondary"),
    ],
    home_on_close=True,
)

## Home the transport and disconnect

Homing drives the transport to its home position and confirms it arrived. Do it before a person
reaches for the plate, unless the last batch already closed with `home_on_close=True`.

`stop()` closes the link. It does nothing on an instrument that is already closed, so it is safe to
run this cell twice, and it is worth running from a `finally` in a script so that a failed run does
not leave the port open.

In [ ]:
await device.home()
await device.stop()

```{note}
The fluid left in the syringes and the cassettes after a run is the instrument's problem, not the
driver's. Follow the manufacturer's shutdown and maintenance procedure — purging each pump
(`device.peristaltic_dispenser.purge(...)`) is what empties a cassette before it comes out, and most
maintenance routines ship as protocol files you can run with `run_protocol()`.
```